# Significance test

### Logirstic Regression baseline multiseed vs network mulitseed

In [1]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\2_Logistic_Regression_model\results")

with open(base / "results_lr_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_lr_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05 

def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list])

def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []
    for metric in METRICS:
        a = extract(no_data[key],   metric)   
        b = extract(with_data[key], metric)   
        diff = b - a                           

        # Paired t-test (n=3, so df=2)
        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric":   metric,
            "mean_no":  a.mean(),
            "mean_with":b.mean(),
            "diff":     diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t":        t_stat,
            "p":        p_val,
            "sig":      p_val < ALPHA,
        })
    return results

def print_table(results, title):
    print(f"\n{'═'*78}")
    print(f"  {title}")
    print(f"{'═'*78}")
    header = f"{'Metric':<14} {'no_net':>8} {'with_net':>9} {'Δ (with−no)':>12} {'t':>7} {'p':>8}  sig?"
    print(header)
    print("─" * 78)
    for r in results:
        flag  = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""
        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p']:>8.4f}"
            f"  {flag}"
        )
    print("─" * 78)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant at α={ALPHA}: {n_sig}/{len(results)} metrics")

for split in ("val", "test"):
    print_table(run_tests(lr_no, lr_with, split), f"Logistic Regression — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a
    return diff.mean() / diff.std(ddof=1)

def interpret(d):
    ad = abs(d)
    if ad < 0.2:  return "negligible"
    if ad < 0.5:  return "small"
    if ad < 0.8:  return "medium"
    return "large"

for model_name, no_data, with_data in [
    ("LR", lr_no, lr_with),
]:
    for split in ("val", "test"):
        res = run_tests(no_data, with_data, split)
        for r in res:
            if r["sig"]:
                d = cohens_d(no_data, with_data, split, r["metric"])
                print(f"{model_name:<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════
  Logistic Regression — VAL
══════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t        p  sig?
──────────────────────────────────────────────────────────────────────────────
HitRate@1        0.1616    0.7749 +   +0.6134 1068.797   0.0000  ✓ YES
HitRate@5        0.5689    0.9168 +   +0.3479 766.137   0.0000  ✓ YES
HitRate@10       0.7480    0.9452 +   +0.1972 312.887   0.0000  ✓ YES
HitRate@20       0.8434    0.9679 +   +0.1244 425.383   0.0000  ✓ YES
NDCG@1           0.1616    0.7749 +   +0.6134 1068.797   0.0000  ✓ YES
NDCG@5           0.3669    0.8554 +   +0.4886 2772.879   0.0000  ✓ YES
NDCG@10          0.4257    0.8647 +   +0.4390 3032.001   0.0000  ✓ YES
NDCG@20          0.4499    0.8704 +   +0.4205 3002.435   0.0000  ✓ YES
MRR              0.3366    0.8409 +   +0.5044 1727.467   0.0000  ✓ YES
────────────────────────────

### Random Forest baseline multiseed vs network mulitseed

In [3]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\3_Random_forest_model\results")

with open(base / "results_rf_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_rf_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05 

def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list])

def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []
    for metric in METRICS:
        a = extract(no_data[key],   metric)   
        b = extract(with_data[key], metric)   
        diff = b - a                           

        # Paired t-test (n=3, so df=2)
        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric":   metric,
            "mean_no":  a.mean(),
            "mean_with":b.mean(),
            "diff":     diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t":        t_stat,
            "p":        p_val,
            "sig":      p_val < ALPHA,
        })
    return results

def print_table(results, title):
    print(f"\n{'═'*78}")
    print(f"  {title}")
    print(f"{'═'*78}")
    header = f"{'Metric':<14} {'no_net':>8} {'with_net':>9} {'Δ (with−no)':>12} {'t':>7} {'p':>8}  sig?"
    print(header)
    print("─" * 78)
    for r in results:
        flag  = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""
        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p']:>8.4f}"
            f"  {flag}"
        )
    print("─" * 78)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant at α={ALPHA}: {n_sig}/{len(results)} metrics")

for split in ("val", "test"):
    print_table(run_tests(lr_no, lr_with, split), f"Random Forest — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a
    return diff.mean() / diff.std(ddof=1)

def interpret(d):
    ad = abs(d)
    if ad < 0.2:  return "negligible"
    if ad < 0.5:  return "small"
    if ad < 0.8:  return "medium"
    return "large"

for model_name, no_data, with_data in [
    ("RF", lr_no, lr_with),
]:
    for split in ("val", "test"):
        res = run_tests(no_data, with_data, split)
        for r in res:
            if r["sig"]:
                d = cohens_d(no_data, with_data, split, r["metric"])
                print(f"{model_name:<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════
  Random Forest — VAL
══════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t        p  sig?
──────────────────────────────────────────────────────────────────────────────
HitRate@1        0.6204    0.9587 +   +0.3383  92.325   0.0001  ✓ YES
HitRate@5        0.7775    0.9723 +   +0.1948 168.439   0.0000  ✓ YES
HitRate@10       0.8347    0.9788 +   +0.1442 125.295   0.0001  ✓ YES
HitRate@20       0.8991    0.9869 +   +0.0878 214.187   0.0000  ✓ YES
NDCG@1           0.6204    0.9587 +   +0.3383  92.325   0.0001  ✓ YES
NDCG@5           0.7056    0.9659 +   +0.2604 154.264   0.0000  ✓ YES
NDCG@10          0.7240    0.9681 +   +0.2441 155.701   0.0000  ✓ YES
NDCG@20          0.7403    0.9701 +   +0.2298 151.276   0.0000  ✓ YES
MRR              0.6966    0.9656 +   +0.2691 124.855   0.0001  ✓ YES
────────────────────────────────────────

### XGBoost baseline multiseed vs network mulitseed

In [2]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\4_XGBoost_model\results")

with open(base / "results_xgb_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_xgb_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05 

def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list])

def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []
    for metric in METRICS:
        a = extract(no_data[key],   metric)   
        b = extract(with_data[key], metric)   
        diff = b - a                           

        # Paired t-test (n=3, so df=2)
        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric":   metric,
            "mean_no":  a.mean(),
            "mean_with":b.mean(),
            "diff":     diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t":        t_stat,
            "p":        p_val,
            "sig":      p_val < ALPHA,
        })
    return results

def print_table(results, title):
    print(f"\n{'═'*78}")
    print(f"  {title}")
    print(f"{'═'*78}")
    header = f"{'Metric':<14} {'no_net':>8} {'with_net':>9} {'Δ (with−no)':>12} {'t':>7} {'p':>8}  sig?"
    print(header)
    print("─" * 78)
    for r in results:
        flag  = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""
        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p']:>8.4f}"
            f"  {flag}"
        )
    print("─" * 78)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant at α={ALPHA}: {n_sig}/{len(results)} metrics")

for split in ("val", "test"):
    print_table(run_tests(lr_no, lr_with, split), f"XGBoost — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a
    return diff.mean() / diff.std(ddof=1)

def interpret(d):
    ad = abs(d)
    if ad < 0.2:  return "negligible"
    if ad < 0.5:  return "small"
    if ad < 0.8:  return "medium"
    return "large"

for model_name, no_data, with_data in [
    ("XGBoost", lr_no, lr_with),
]:
    for split in ("val", "test"):
        res = run_tests(no_data, with_data, split)
        for r in res:
            if r["sig"]:
                d = cohens_d(no_data, with_data, split, r["metric"])
                print(f"{model_name:<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════
  XGBoost — VAL
══════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t        p  sig?
──────────────────────────────────────────────────────────────────────────────
HitRate@1        0.7744    0.9949 +   +0.2205 384.457   0.0000  ✓ YES
HitRate@5        0.8625    0.9970 +   +0.1345  80.937   0.0002  ✓ YES
HitRate@10       0.9002    0.9977 +   +0.0974  60.909   0.0003  ✓ YES
HitRate@20       0.9399    0.9984 +   +0.0585  57.018   0.0003  ✓ YES
NDCG@1           0.7744    0.9949 +   +0.2205 384.457   0.0000  ✓ YES
NDCG@5           0.8214    0.9960 +   +0.1746 267.830   0.0000  ✓ YES
NDCG@10          0.8336    0.9962 +   +0.1627 452.135   0.0000  ✓ YES
NDCG@20          0.8436    0.9964 +   +0.1528 613.490   0.0000  ✓ YES
MRR              0.8172    0.9959 +   +0.1787 660.945   0.0000  ✓ YES
──────────────────────────────────────────────

### Neural Network baseline multiseed vs network mulitseed

In [2]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\5_neural_Network_model\results")

with open(base / "results_NN_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_NN_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05 

def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list])

def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []
    for metric in METRICS:
        a = extract(no_data[key],   metric)   
        b = extract(with_data[key], metric)   
        diff = b - a                           

        # Paired t-test (n=3, so df=2)
        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric":   metric,
            "mean_no":  a.mean(),
            "mean_with":b.mean(),
            "diff":     diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t":        t_stat,
            "p":        p_val,
            "sig":      p_val < ALPHA,
        })
    return results

def print_table(results, title):
    print(f"\n{'═'*78}")
    print(f"  {title}")
    print(f"{'═'*78}")
    header = f"{'Metric':<14} {'no_net':>8} {'with_net':>9} {'Δ (with−no)':>12} {'t':>7} {'p':>8}  sig?"
    print(header)
    print("─" * 78)
    for r in results:
        flag  = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""
        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p']:>8.4f}"
            f"  {flag}"
        )
    print("─" * 78)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant at α={ALPHA}: {n_sig}/{len(results)} metrics")

for split in ("val", "test"):
    print_table(run_tests(lr_no, lr_with, split), f"NN — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a
    return diff.mean() / diff.std(ddof=1)

def interpret(d):
    ad = abs(d)
    if ad < 0.2:  return "negligible"
    if ad < 0.5:  return "small"
    if ad < 0.8:  return "medium"
    return "large"

for model_name, no_data, with_data in [
    ("NN", lr_no, lr_with),
]:
    for split in ("val", "test"):
        res = run_tests(no_data, with_data, split)
        for r in res:
            if r["sig"]:
                d = cohens_d(no_data, with_data, split, r["metric"])
                print(f"{model_name:<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════
  NN — VAL
══════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t        p  sig?
──────────────────────────────────────────────────────────────────────────────
HitRate@1        0.1848    0.1826    -0.0022  -1.932   0.1931    no
HitRate@5        0.4871    0.4904 +   +0.0033   0.690   0.5616    no
HitRate@10       0.6859    0.6886 +   +0.0027   0.557   0.6337    no
HitRate@20       0.8767    0.8801 +   +0.0034   1.745   0.2231    no
NDCG@1           0.1848    0.1826    -0.0022  -1.932   0.1931    no
NDCG@5           0.3382    0.3385 +   +0.0004   0.149   0.8949    no
NDCG@10          0.4023    0.4026 +   +0.0002   0.091   0.9358    no
NDCG@20          0.4508    0.4512 +   +0.0004   0.210   0.8534    no
MRR              0.3334    0.3327    -0.0006  -0.371   0.7461    no
───────────────────────────────────────────────────────────────

### GraphSAGE baseline multiseed vs network mulitseed

In [3]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\6_GraphSAGE_model\results")

with open(base / "results_GraphSAGE_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_GraphSAGE_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05 

def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list])

def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []
    for metric in METRICS:
        a = extract(no_data[key],   metric)   
        b = extract(with_data[key], metric)   
        diff = b - a                           

        # Paired t-test (n=3, so df=2)
        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric":   metric,
            "mean_no":  a.mean(),
            "mean_with":b.mean(),
            "diff":     diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t":        t_stat,
            "p":        p_val,
            "sig":      p_val < ALPHA,
        })
    return results

def print_table(results, title):
    print(f"\n{'═'*78}")
    print(f"  {title}")
    print(f"{'═'*78}")
    header = f"{'Metric':<14} {'no_net':>8} {'with_net':>9} {'Δ (with−no)':>12} {'t':>7} {'p':>8}  sig?"
    print(header)
    print("─" * 78)
    for r in results:
        flag  = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""
        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p']:>8.4f}"
            f"  {flag}"
        )
    print("─" * 78)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant at α={ALPHA}: {n_sig}/{len(results)} metrics")

for split in ("val", "test"):
    print_table(run_tests(lr_no, lr_with, split), f"GraphSAGE — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a
    return diff.mean() / diff.std(ddof=1)

def interpret(d):
    ad = abs(d)
    if ad < 0.2:  return "negligible"
    if ad < 0.5:  return "small"
    if ad < 0.8:  return "medium"
    return "large"

for model_name, no_data, with_data in [
    ("GraphSAGE", lr_no, lr_with),
]:
    for split in ("val", "test"):
        res = run_tests(no_data, with_data, split)
        for r in res:
            if r["sig"]:
                d = cohens_d(no_data, with_data, split, r["metric"])
                print(f"{model_name:<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════
  GraphSAGE — VAL
══════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t        p  sig?
──────────────────────────────────────────────────────────────────────────────
HitRate@1        0.1971    0.2017 +   +0.0046   1.248   0.3383    no
HitRate@5        0.5366    0.5424 +   +0.0058   2.323   0.1458    no
HitRate@10       0.7329    0.7375 +   +0.0045   2.377   0.1406    no
HitRate@20       0.9026    0.9049 +   +0.0023   1.853   0.2051    no
NDCG@1           0.1971    0.2017 +   +0.0046   1.248   0.3383    no
NDCG@5           0.3702    0.3754 +   +0.0052   1.588   0.2531    no
NDCG@10          0.4338    0.4385 +   +0.0047   1.528   0.2660    no
NDCG@20          0.4770    0.4811 +   +0.0041   1.401   0.2962    no
MRR              0.3573    0.3618 +   +0.0045   1.321   0.3175    no
─────────────────────────────────────────────────────

# Testing simulations

## XGBoost baseline vs baseline simulation 

In [1]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models")

xgb_baseline_path = base / "4_XGBoost_model" / "results" / "results_xgb_baseline_multiseed.json"
xgb_sim_path = base / "9_simulation_xgb" / "results" / "results_xgb_baseline_sim_multiseed.json"

with open(xgb_baseline_path, "r", encoding="utf-8") as f:
    xgb_baseline = json.load(f)

with open(xgb_sim_path, "r", encoding="utf-8") as f:
    xgb_sim = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1", "NDCG@5", "NDCG@10", "NDCG@20",
    "MRR",
]

ALPHA = 0.05

def extract(data, split, metric):
    key = f"per_seed_{split}"
    return np.array([seed_result[metric] for seed_result in data[key]], dtype=float)

def run_tests(original_data, simulation_data, split="test"):
    results = []

    for metric in METRICS:
        original = extract(original_data, split, metric)
        simulation = extract(simulation_data, split, metric)

        diff = simulation - original

        t_stat, p_val = stats.ttest_rel(simulation, original)

        results.append({
            "metric": metric,
            "mean_original": original.mean(),
            "mean_simulation": simulation.mean(),
            "diff": diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t": t_stat,
            "p": p_val,
            "sig": p_val < ALPHA,
        })

    return results

def print_table(results, title):
    print(f"\n{'═' * 90}")
    print(f"  {title}")
    print(f"{'═' * 90}")
    print(
        f"{'Metric':<14} "
        f"{'Original':>10} "
        f"{'Simulation':>12} "
        f"{'Δ(sim-orig)':>13} "
        f"{'t':>9} "
        f"{'p':>10} "
        f"sig?"
    )
    print("─" * 90)

    for r in results:
        flag = "✓ YES" if r["sig"] else "no"
        print(
            f"{r['metric']:<14} "
            f"{r['mean_original']:>10.4f} "
            f"{r['mean_simulation']:>12.4f} "
            f"{r['diff']:>+13.4f} "
            f"{r['t']:>9.3f} "
            f"{r['p']:>10.4f} "
            f"{flag}"
        )

    print("─" * 90)
    n_sig = sum(r["sig"] for r in results)
    print(f"Significant at α={ALPHA}: {n_sig}/{len(results)} metrics")

def cohens_d_paired(original_data, simulation_data, split, metric):
    original = extract(original_data, split, metric)
    simulation = extract(simulation_data, split, metric)
    diff = simulation - original

    std = diff.std(ddof=1)
    if std == 0:
        return np.nan

    return diff.mean() / std

def interpret_d(d):
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    if ad < 0.5:
        return "small"
    if ad < 0.8:
        return "medium"
    return "large"

all_results = {}

for split in ("val", "test"):
    results = run_tests(xgb_baseline, xgb_sim, split)
    all_results[split] = results
    print_table(results, f"XGBoost Baseline vs Simulation — {split.upper()}")

print("\n\nEffect sizes for significant results:")
print(f"{'Split':<6} {'Metric':<14} {'Cohen d':>10}  interpretation")
print("─" * 50)

for split in ("val", "test"):
    for r in all_results[split]:
        if r["sig"]:
            d = cohens_d_paired(xgb_baseline, xgb_sim, split, r["metric"])
            print(f"{split:<6} {r['metric']:<14} {d:>10.3f}  {interpret_d(d)}")


══════════════════════════════════════════════════════════════════════════════════════════
  XGBoost Baseline vs Simulation — VAL
══════════════════════════════════════════════════════════════════════════════════════════
Metric           Original   Simulation   Δ(sim-orig)         t          p sig?
──────────────────────────────────────────────────────────────────────────────────────────
HitRate@1          0.7744       0.7757       +0.0013     0.390     0.7341 no
HitRate@5          0.8625       0.8635       +0.0010     0.438     0.7043 no
HitRate@10         0.9002       0.9045       +0.0043     1.613     0.2480 no
HitRate@20         0.9399       0.9405       +0.0006     0.682     0.5658 no
NDCG@1             0.7744       0.7757       +0.0013     0.390     0.7341 no
NDCG@5             0.8214       0.8222       +0.0009     0.311     0.7852 no
NDCG@10            0.8336       0.8354       +0.0019     0.720     0.5463 no
NDCG@20            0.8436       0.8446       +0.0010     0.404     0.

## XGBoost network vs network simulation 

In [3]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models")

xgb_network_path = base / "4_XGBoost_model" / "results" / "results_xgb_network_multiseed.json"
xgb_sim_path = base / "9_simulation_xgb" / "results" / "results_xgb_network_sim_multiseed.json"

with open(xgb_network_path, "r", encoding="utf-8") as f:
    xgb_network = json.load(f)

with open(xgb_sim_path, "r", encoding="utf-8") as f:
    xgb_sim = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1", "NDCG@5", "NDCG@10", "NDCG@20",
    "MRR",
]

ALPHA = 0.05

def extract(data, split, metric):
    key = f"per_seed_{split}"
    return np.array([seed_result[metric] for seed_result in data[key]], dtype=float)

def run_tests(original_data, simulation_data, split="test"):
    results = []

    for metric in METRICS:
        original = extract(original_data, split, metric)
        simulation = extract(simulation_data, split, metric)

        diff = simulation - original

        t_stat, p_val = stats.ttest_rel(simulation, original)

        results.append({
            "metric": metric,
            "mean_original": original.mean(),
            "mean_simulation": simulation.mean(),
            "diff": diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t": t_stat,
            "p": p_val,
            "sig": p_val < ALPHA,
        })

    return results

def print_table(results, title):
    print(f"\n{'═' * 90}")
    print(f"  {title}")
    print(f"{'═' * 90}")
    print(
        f"{'Metric':<14} "
        f"{'Original':>10} "
        f"{'Simulation':>12} "
        f"{'Δ(sim-orig)':>13} "
        f"{'t':>9} "
        f"{'p':>10} "
        f"sig?"
    )
    print("─" * 90)

    for r in results:
        flag = "✓ YES" if r["sig"] else "no"
        print(
            f"{r['metric']:<14} "
            f"{r['mean_original']:>10.4f} "
            f"{r['mean_simulation']:>12.4f} "
            f"{r['diff']:>+13.4f} "
            f"{r['t']:>9.3f} "
            f"{r['p']:>10.4f} "
            f"{flag}"
        )

    print("─" * 90)
    n_sig = sum(r["sig"] for r in results)
    print(f"Significant at α={ALPHA}: {n_sig}/{len(results)} metrics")

def cohens_d_paired(original_data, simulation_data, split, metric):
    original = extract(original_data, split, metric)
    simulation = extract(simulation_data, split, metric)
    diff = simulation - original

    std = diff.std(ddof=1)
    if std == 0:
        return np.nan

    return diff.mean() / std

def interpret_d(d):
    ad = abs(d)
    if ad < 0.2:
        return "negligible"
    if ad < 0.5:
        return "small"
    if ad < 0.8:
        return "medium"
    return "large"

all_results = {}

for split in ("val", "test"):
    results = run_tests(xgb_network, xgb_sim, split)
    all_results[split] = results
    print_table(results, f"XGBoost Network vs Simulation — {split.upper()}")

print("\n\nEffect sizes for significant results:")
print(f"{'Split':<6} {'Metric':<14} {'Cohen d':>10}  interpretation")
print("─" * 50)

for split in ("val", "test"):
    for r in all_results[split]:
        if r["sig"]:
            d = cohens_d_paired(xgb_network, xgb_sim, split, r["metric"])
            print(f"{split:<6} {r['metric']:<14} {d:>10.3f}  {interpret_d(d)}")


══════════════════════════════════════════════════════════════════════════════════════════
  XGBoost Network vs Simulation — VAL
══════════════════════════════════════════════════════════════════════════════════════════
Metric           Original   Simulation   Δ(sim-orig)         t          p sig?
──────────────────────────────────────────────────────────────────────────────────────────
HitRate@1          0.9949       0.9933       -0.0016   -10.122     0.0096 ✓ YES
HitRate@5          0.9970       0.9967       -0.0003    -1.685     0.2341 no
HitRate@10         0.9977       0.9975       -0.0002    -4.044     0.0560 no
HitRate@20         0.9984       0.9983       -0.0001    -0.519     0.6556 no
NDCG@1             0.9949       0.9933       -0.0016   -10.122     0.0096 ✓ YES
NDCG@5             0.9960       0.9950       -0.0010    -8.818     0.0126 ✓ YES
NDCG@10            0.9962       0.9953       -0.0010   -12.963     0.0059 ✓ YES
NDCG@20            0.9964       0.9955       -0.0010   -14

## XGBoost variation 

In [6]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

# =========================
# Paths
# =========================
base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models")

xgb_var = base / "8_xgb_variations" / "results" / "multiseed"
xgb_main = base / "4_XGBoost_model" / "results"

paths = {
    "baseline": xgb_main / "results_xgb_baseline_multiseed.json",
    "network": xgb_main / "results_xgb_network_multiseed.json",

    "baseline_emb0": xgb_var / "results_xgb_baseline+emb_0_multiseed.json",
    "embeddings_only": xgb_var / "results_embeddings_only_multiseed.json",
    "embeddings_basic": xgb_var / "results_xgb_embeddings+unique_genres_played+total_playtime_minutes_multiseed.json",

    "baseline_no_user_count": xgb_var / "results_xgb_baseline-user_count_multiseed.json",
    "baseline_emb0_no_user_count": xgb_var / "results_xgb_baseline+emb_0-user_count_multiseed.json",

    "user_count_emb0_only": xgb_var / "results_xgb_emb0+user_count_multiseed.json",

    "baseline_log_user_count": xgb_var / "results_xgb_baseline_log(user_count)_multiseed.json",
    "baseline_fake_emb0": xgb_var / "results_xgb_baseline+fake_emb_0_multiseed.json",
}

models = {}
for name, path in paths.items():
    with open(path, "r", encoding="utf-8") as f:
        models[name] = json.load(f)

# =========================
# Settings
# =========================
METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1", "NDCG@5", "NDCG@10", "NDCG@20",
    "MRR",
]

ALPHA = 0.05

COMPARISONS = [
    ("baseline", "baseline_emb0", "Effect of adding game_emb_0 to baseline"),
    ("baseline", "embeddings_only", "Embeddings only compared to full baseline"),
    ("embeddings_only", "embeddings_basic", "Effect of adding basic user info to embeddings"),
    ("baseline", "baseline_no_user_count", "Effect of removing user_count"),
    ("baseline_no_user_count", "baseline_emb0_no_user_count", "Effect of emb0 without user_count"),
    ("baseline", "user_count_emb0_only", "Minimal model: user_count + emb0 vs baseline"),
    ("user_count_emb0_only", "network", "Minimal model vs full network"),
    ("baseline", "baseline_log_user_count", "Effect of log(user_count)"),
    ("baseline", "baseline_fake_emb0", "Effect of fake emb0"),
    ("baseline_emb0", "baseline_fake_emb0", "Real emb0 vs fake emb0"),
]

# =========================
# Helpers
# =========================
def extract(data, split, metric):
    key = f"per_seed_{split}"
    rows = sorted(data[key], key=lambda x: x["seed"])
    return np.array([r[metric] for r in rows], dtype=float)

def paired_test(a, b):
    diff = b - a

    if np.allclose(diff.std(ddof=1), 0):
        if np.allclose(diff.mean(), 0):
            return 0.0, 1.0, diff
        else:
            return np.inf * np.sign(diff.mean()), 0.0, diff

    t_stat, p_val = stats.ttest_rel(b, a)
    return float(t_stat), float(p_val), diff

def holm_correction(pvals):
    """
    Holm-Bonferroni correction.
    Returns adjusted p-values in original order.
    """
    pvals = np.array(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adjusted = np.empty(m)

    prev = 0
    for i, idx in enumerate(order):
        adj = (m - i) * pvals[idx]
        adj = max(adj, prev)
        adjusted[idx] = min(adj, 1.0)
        prev = adjusted[idx]

    return adjusted

def run_comparison(model_a, model_b, split="test"):
    raw_results = []

    for metric in METRICS:
        a = extract(models[model_a], split, metric)
        b = extract(models[model_b], split, metric)

        t_stat, p_val, diff = paired_test(a, b)

        raw_results.append({
            "metric": metric,
            "mean_a": a.mean(),
            "mean_b": b.mean(),
            "diff": diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t": t_stat,
            "p": p_val,
        })

    corrected = holm_correction([r["p"] for r in raw_results])

    for r, p_adj in zip(raw_results, corrected):
        r["p_holm"] = p_adj
        r["sig_raw"] = r["p"] < ALPHA
        r["sig_holm"] = r["p_holm"] < ALPHA

    return raw_results

def print_comparison(model_a, model_b, title, split="test"):
    results = run_comparison(model_a, model_b, split)

    print(f"\n{'═' * 105}")
    print(f"{title} — {split.upper()}")
    print(f"{model_a}  ->  {model_b}")
    print(f"{'═' * 105}")
    print(
        f"{'Metric':<14} "
        f"{model_a[:15]:>12} "
        f"{model_b[:15]:>12} "
        f"{'Δ':>10} "
        f"{'t':>9} "
        f"{'p':>10} "
        f"{'p_holm':>10} "
        f"sig?"
    )
    print("─" * 105)

    for r in results:
        sig = "✓ YES" if r["sig_holm"] else "no"
        print(
            f"{r['metric']:<14} "
            f"{r['mean_a']:>12.4f} "
            f"{r['mean_b']:>12.4f} "
            f"{r['diff']:>+10.4f} "
            f"{r['t']:>9.3f} "
            f"{r['p']:>10.4g} "
            f"{r['p_holm']:>10.4g} "
            f"{sig}"
        )

    print("─" * 105)
    print(f"Significant after Holm correction: {sum(r['sig_holm'] for r in results)}/{len(results)}")

# =========================
# Run all comparisons
# =========================
for split in ("val", "test"):
    for a, b, title in COMPARISONS:
        print_comparison(a, b, title, split)


═════════════════════════════════════════════════════════════════════════════════════════════════════════
Effect of adding game_emb_0 to baseline — VAL
baseline  ->  baseline_emb0
═════════════════════════════════════════════════════════════════════════════════════════════════════════
Metric             baseline baseline_emb0          Δ         t          p     p_holm sig?
─────────────────────────────────────────────────────────────────────────────────────────────────────────
HitRate@1            0.7744       0.9841    +0.2097   296.932  1.134e-05  6.359e-05 ✓ YES
HitRate@5            0.8625       0.9915    +0.1290    90.996  0.0001207  0.0003622 ✓ YES
HitRate@10           0.9002       0.9944    +0.0942    64.341  0.0002415  0.0004829 ✓ YES
HitRate@20           0.9399       0.9963    +0.0564    53.811  0.0003452  0.0004829 ✓ YES
NDCG@1               0.7744       0.9841    +0.2097   296.932  1.134e-05  6.359e-05 ✓ YES
NDCG@5               0.8214       0.9880    +0.1666   307.167   1.0


═════════════════════════════════════════════════════════════════════════════════════════════════════════
Effect of log(user_count) — TEST
baseline  ->  baseline_log_user_count
═════════════════════════════════════════════════════════════════════════════════════════════════════════
Metric             baseline baseline_log_us          Δ         t          p     p_holm sig?
─────────────────────────────────────────────────────────────────────────────────────────────────────────
HitRate@1            0.7768       0.7730    -0.0039    -1.454     0.2832          1 no
HitRate@5            0.8630       0.8591    -0.0039    -1.104     0.3845          1 no
HitRate@10           0.8992       0.8965    -0.0027    -1.567     0.2577          1 no
HitRate@20           0.9382       0.9370    -0.0012    -0.650     0.5825          1 no
NDCG@1               0.7768       0.7730    -0.0039    -1.454     0.2832          1 no
NDCG@5               0.8228       0.8188    -0.0040    -1.230     0.3438          1